# Sesión 01 — Modelado del Canal Inalámbrico
### Laboratorio Python · Sistemas de Comunicaciones Inalámbricas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/01-channel-modeling/lab.ipynb)

---
> **Licencia**: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) · *Sistemas de Comunicaciones Inalámbricas* por ollerenac

## Objetivos del Laboratorio

Al completar este laboratorio serás capaz de:

1. Simular la envolvente de desvanecimiento Rayleigh y verificar la distribución teórica.
2. Comparar distribuciones de amplitud Rician para distintos valores del factor K.
3. Calcular curvas de BER para BPSK sobre AWGN y canal Rayleigh y cuantificar la penalización.
4. Calcular la CDF empírica de la envolvente y estimar la probabilidad de interrupción.
5. Caracterizar un canal multicamino real: dispersión de retardo, ancho de banda coherente, tiempo coherente y BER en escenarios urbano y rural.
6. Aplicar los parámetros de TR 38.901 para caracterizar completamente un canal 5G NR.

In [ ]:
%pip install numpy scipy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rayleigh, rice
from scipy.special import i0, i0e

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
rng = np.random.default_rng(seed=42)

## Repaso Teórico

### De los caminos multitrayecto a las componentes I/Q

En el index.md de esta sesión derivamos paso a paso el origen de $I$ y $Q$. El resultado clave es:

cada camino $i$ contribuye $a_i\cos(2\pi f_c t + \phi_i)$. Aplicando la identidad $\cos(\alpha+\beta) = \cos\alpha\cos\beta - \sin\alpha\sin\beta$, esa contribución se descompone en:

$$a_i\cos\phi_i \cdot \cos(2\pi f_c t) \;-\; a_i\sin\phi_i \cdot \sin(2\pi f_c t)$$

Sumando los $N$ caminos, la señal total tiene dos coeficientes:

$$I = \sum_{i=1}^N a_i\cos\phi_i \qquad Q = \sum_{i=1}^N a_i\sin\phi_i$$

Por el teorema central del límite, $I, Q \sim \mathcal{N}(0, \sigma^2)$ cuando $N$ es grande y las fases $\phi_i$ son uniformes en $[0, 2\pi)$. El coeficiente de canal en banda base es:

$$h = I + j\,Q$$

### Desvanecimiento Rayleigh

La envolvente $r = |h| = \sqrt{I^2 + Q^2}$ sigue una distribución Rayleigh:

$$f_R(r) = \frac{r}{\sigma^2}\,e^{-r^2/(2\sigma^2)}, \quad r \geq 0$$

La BER de BPSK en canal Rayleigh plano (promediada sobre el fading) es:

$$\overline{\text{BER}}_\text{Rayleigh} = \frac{1}{2}\left(1 - \sqrt{\frac{\bar{\gamma}}{2+\bar{\gamma}}}\right) \approx \frac{1}{2\bar{\gamma}} \quad (\bar{\gamma} \gg 1)$$

donde $\bar{\gamma} = P_r/P_n = A^2/\sigma_n^2$ es la **SNR media** — la misma convención de §6: cociente entre potencia de señal y potencia de ruido en el receptor.

### Desvanecimiento Rician

Cuando existe una componente LOS con amplitud fija $A$, las medias de $I$ y $Q$ se desplazan del origen: $I \sim \mathcal{N}(A\cos\phi_\text{LOS}, \sigma^2)$, $Q \sim \mathcal{N}(A\sin\phi_\text{LOS}, \sigma^2)$. La envolvente sigue una distribución Rice:

$$f_R(r) = \frac{r}{\sigma^2}\,e^{-(r^2+A^2)/(2\sigma^2)}\,I_0\!\left(\frac{rA}{\sigma^2}\right), \quad r \geq 0$$

El **factor K** es la relación entre la potencia LOS ($A^2/2$) y la potencia de las componentes dispersas ($2\sigma^2$):

$$K = \frac{A^2}{2\sigma^2}$$

$K=0$ corresponde a Rayleigh puro ($A=0$, NLOS); $K\to\infty$ equivale a canal AWGN sin fading.

---
## Ejercicio 1 — Desvanecimiento Rayleigh: simulación y verificación de la PDF

Generaremos las componentes en fase y cuadratura de un canal Rayleigh plano y comprobaremos que la envolvente simulada coincide con la PDF teórica de Rayleigh.

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
N = 100_000
sigma = 1 / np.sqrt(2)  # desviación típica de cada componente → E[r²] = 2σ² = 1

# ── Simulación: I y Q son gaussianas IID con media cero ─────────────────────
# (representa N → ∞ caminos con fases uniformes; cada muestra es una
#  realización independiente del canal — no incluye correlación temporal)
I_comp = rng.normal(0, sigma, N)
Q_comp = rng.normal(0, sigma, N)
r = np.abs(I_comp + 1j * Q_comp)   # envolvente = |h| = sqrt(I²+Q²)

# ── PDF teórica de Rayleigh con σ = 1/√2 ────────────────────────────────────
r_axis = np.linspace(0, 4, 500)
pdf_theory = rayleigh.pdf(r_axis, scale=sigma)

# ── Figura ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Serie temporal de la envolvente (primeras 500 muestras)
axes[0].plot(r[:500], lw=0.8, color='steelblue')
axes[0].set_xlabel('Índice de muestra')
axes[0].set_ylabel('Envolvente $r$')
axes[0].set_title('Envolvente Rayleigh — primeras 500 muestras')

# (b) Histograma vs PDF teórica
axes[1].hist(r, bins=80, density=True, alpha=0.6, color='steelblue', label='Simulada')
axes[1].plot(r_axis, pdf_theory, 'r-', lw=2, label='Teórica Rayleigh')
axes[1].set_xlabel('Envolvente $r$')
axes[1].set_ylabel('Densidad de probabilidad')
axes[1].set_title('PDF de la envolvente Rayleigh')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Potencia media simulada : {np.mean(r**2):.4f}  (esperada: {2*sigma**2:.4f})')
print(f'Valor cuadrático medio  : {np.sqrt(np.mean(r**2)):.4f}')

**Preguntas de reflexión:**

1. ¿Por qué la envolvente nunca es negativa aunque $I$ y $Q$ sí lo son?
2. ¿Qué ocurre con la forma de la PDF si aumentas $\sigma$? Prueba con $\sigma = 0.5$ y $\sigma = 2$.
3. ¿Cuál es la probabilidad de que la envolvente caiga por debajo de $-10\,\text{dB}$ (es decir, $r < 10^{-10/20}$)? Calcula con la CDF teórica: $F_R(r) = 1 - e^{-r^2/(2\sigma^2)}$.

---
## Ejercicio 2 — Desvanecimiento Rician: efecto del factor K

Simularemos y compararemos las PDFs de la envolvente para K = 0, 1, 5, 10 dB.

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
N = 200_000
K_labels = ['K = -∞ (Rayleigh)', 'K = 0 dB', 'K = 5 dB', 'K = 10 dB']
K_lin    = [0, 1, 10**0.5, 10]      # K en lineal
colors   = ['steelblue', 'darkorange', 'green', 'red']

r_axis = np.linspace(0, 3.5, 500)

fig, ax = plt.subplots(figsize=(9, 5))

for K, label, color in zip(K_lin, K_labels, colors):
    # Potencia total unitaria: A²/2 + 2σ² = 1
    #   → σ² = 1 / (2(K+1)),   A = √(K/(K+1))
    sigma2 = 1 / (2 * (K + 1))
    sigma  = np.sqrt(sigma2)
    A      = np.sqrt(K / (K + 1))   # amplitud de la componente LOS

    # Simulación: componente LOS desplaza la media de I
    # (φ_LOS = 0 → μ_I = A, μ_Q = 0)
    I_comp = rng.normal(A,     sigma, N)
    Q_comp = rng.normal(0.0,   sigma, N)
    r_sim  = np.abs(I_comp + 1j * Q_comp)

    # PDF teórica de Rice: scale=σ, b=A/σ (parámetro de no-centralidad normalizado)
    if sigma > 0:
        pdf_th = rice.pdf(r_axis, b=A / sigma, scale=sigma)
    else:
        pdf_th = np.zeros_like(r_axis)

    ax.hist(r_sim, bins=100, density=True, alpha=0.25, color=color)
    ax.plot(r_axis, pdf_th, color=color, lw=2, label=label)

ax.set_xlabel('Envolvente $r$')
ax.set_ylabel('Densidad de probabilidad')
ax.set_title('Distribución de Rice para distintos valores de K')
ax.legend()
plt.tight_layout()
plt.show()

**Preguntas de reflexión:**

1. A medida que $K$ aumenta, la PDF se desplaza hacia la derecha y se estrecha. ¿Qué implica esto para la probabilidad de **deep fading**?
2. Para $K=0$, verifica que la distribución Rice se reduce a Rayleigh. ¿Cómo lo confirma la figura?
3. Un enlace en interiores sin LOS tiene $K \approx 0$; un enlace satélite puede tener $K > 20\,\text{dB}$. ¿Qué consecuencia tiene esta diferencia en el diseño del fade margin del link budget?

---
## Ejercicio 3 — Curvas BER vs SNR: Rayleigh frente a AWGN

Calcularemos y compararemos la BER de BPSK en (convención §6: $\gamma = \text{SNR} = P_r/P_n$):

- Canal AWGN: $\text{BER}^{\text{AWGN}} = P_e^{\text{AWGN}} = Q(\sqrt{\bar{\gamma}})$
- Canal Rayleigh plano (expresión cerrada): $\overline{\text{BER}}^{\text{Ray}} = P_e^{\text{Ray}} = \frac{1}{2}\left(1-\sqrt{\frac{\bar{\gamma}}{2+\bar{\gamma}}}\right) \approx \frac{1}{2\bar{\gamma}}$
- Canal Rayleigh plano (simulación Monte Carlo)

> $P_e$ es la probabilidad de error de bit — lo que a lo largo del curso llamamos **BER**.

**¿Qué es la simulación Monte Carlo aquí?**
En lugar de calcular la BER con una fórmula, la *estimamos empíricamente*: generamos un número grande de bits aleatorios, los transmitimos por un canal Rayleigh simulado (con ruido gaussiano), los detectamos y contamos cuántos llegaron mal. La fracción de bits erróneos es la BER estimada. Repetimos esto para cada valor de $\bar{\gamma}$. Si el número de bits es suficientemente grande ($N \gg 1/\text{BER}$), la estimación converge a la BER teórica.

In [ ]:
from scipy.special import erfc

# ── Rango de SNR (convención §6: γ = P_r/P_n = SNR) ─────────────────────────
SNR_dB  = np.arange(0, 31, 2)
SNR_lin = 10 ** (SNR_dB / 10)

# ── BER teórica AWGN: BER = Q(√γ) ────────────────────────────────────────────
Q = lambda x: 0.5 * erfc(x / np.sqrt(2))
ber_awgn = Q(np.sqrt(SNR_lin))

# ── BER teórica Rayleigh (forma cerrada): BER ≈ 1/(2γ̄) ─────────────────────
ber_rayleigh_th = 0.5 * (1 - np.sqrt(SNR_lin / (2 + SNR_lin)))

# ── Simulación Monte Carlo sobre canal Rayleigh ───────────────────────────────
# Detección coherente BPSK con CSI perfecta:
#   1. h ~ CN(0, 1): canal Rayleigh complejo con potencia media E[|h|²] = 1
#   2. y = h·s + n  (señal recibida)
#   3. Receptor corrige la fase: y_eq = |h|·s + Re(h*/|h|·n)
#   4. Decisión: signo de y_eq
#
# Con γ = SNR = A²/σ_n² y A=1 → σ_n = 1/√γ

N_bits = 200_000
bits   = rng.integers(0, 2, N_bits)
s      = 2 * bits - 1                  # BPSK: {0,1} → {-1,+1}

sigma_channel = 1 / np.sqrt(2)        # E[|h|²] = 2σ² = 1
ber_mc = np.zeros(len(SNR_dB))

for idx, snr in enumerate(SNR_lin):
    h      = (rng.normal(0, sigma_channel, N_bits)
              + 1j * rng.normal(0, sigma_channel, N_bits))
    env    = np.abs(h)
    # γ = 1/σ_n² → σ_n = 1/√γ
    sigma_n = np.sqrt(1.0 / snr)
    noise   = sigma_n * rng.normal(0, 1, N_bits)
    y_eq    = env * s + noise
    bits_rx = (y_eq > 0).astype(int)
    ber_mc[idx] = np.mean(bits_rx != bits)

# ── Figura ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(SNR_dB, ber_awgn,        'b-o',  lw=2, ms=6, label='AWGN (teórica)')
ax.semilogy(SNR_dB, ber_rayleigh_th, 'r-s',  lw=2, ms=6, label='Rayleigh (teórica)')
ax.semilogy(SNR_dB, ber_mc,          'g--^', lw=1.5, ms=6, label='Rayleigh (Monte Carlo)')

ax.set_xlabel('SNR $\\bar{\\gamma}$ (dB)')
ax.set_ylabel('BER')
ax.set_title('BER vs SNR $\\bar{\\gamma}$ — BPSK sobre AWGN y canal Rayleigh plano')
ax.set_ylim([1e-5, 1])
ax.legend()
plt.tight_layout()
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────────
print(f'{"SNR (dB)":>12} {"AWGN":>12} {"Rayleigh (th)":>15} {"Rayleigh (MC)":>15}')
print('-' * 58)
for i, db in enumerate(SNR_dB):
    print(f'{db:>12.0f} {ber_awgn[i]:>12.2e} {ber_rayleigh_th[i]:>15.2e} {ber_mc[i]:>15.2e}')

**Preguntas de reflexión:**

1. A $\bar{\gamma} = 20\,\text{dB}$, ¿cuántos dB se necesitan *extra* sobre el canal Rayleigh para alcanzar la misma BER que en AWGN? Este valor se denomina **penalización por desvanecimiento**.
2. La curva de Rayleigh teórica decae como $1/(2\bar{\gamma})$ a alta SNR, mucho más lento que la gaussiana del AWGN. ¿Por qué esta diferencia fundamental en la pendiente de la curva?
3. ¿Cómo modificarías la simulación para modelar **diversidad de recepción** con dos antenas independientes (MRC, maximal-ratio combining)? Pista: promedia dos coeficientes de canal independientes ponderados por su magnitud.

---
## Ejercicio 4 — CDF empírica y Probabilidad de Interrupción

> **Tiempo estimado: ~15 min**

La **probabilidad de interrupción** (*outage probability*) es la probabilidad de que la potencia recibida caiga por debajo de un umbral mínimo $\gamma_{\text{th}}$:

$$P_{\text{out}}(\gamma_{\text{th}}) = P(R^2 < \gamma_{\text{th}}) = F_R(\sqrt{\gamma_{\text{th}}})$$

donde $F_R(r) = P(R \leq r)$ es la **función de distribución acumulada (CDF)** de la envolvente $R$.

Para un canal Rayleigh con potencia media unitaria:

$$P_{\text{out}}(\gamma_{\text{th}}) = 1 - e^{-\gamma_{\text{th}}}$$

Verificaremos esta expresión con simulación Monte Carlo y la usaremos para calcular el margen de desvanecimiento necesario.

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────────────────
N = 500_000
sigma = 1 / np.sqrt(2)          # potencia media E[R²] = 1

# ── Canal Rayleigh ───────────────────────────────────────────────────────────
I_comp = rng.normal(0, sigma, N)
Q_comp = rng.normal(0, sigma, N)
r   = np.abs(I_comp + 1j * Q_comp)
r2  = r ** 2                    # potencia instantánea

# ── CDF empírica (ordenar muestras) ─────────────────────────────────────────
r2_sorted = np.sort(r2)
cdf_emp   = np.arange(1, N + 1) / N

# ── CDF teórica ──────────────────────────────────────────────────────────────
gamma_axis = np.linspace(0, 5, 500)
cdf_th     = 1 - np.exp(-gamma_axis)   # Rayleigh, potencia media = 1

# ── Figura: CDF ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r2_sorted[::100], cdf_emp[::100], 'b.', ms=2, alpha=0.4, label='Empírica')
axes[0].plot(gamma_axis, cdf_th, 'r-', lw=2, label='Teórica')
axes[0].set_xlabel('Potencia instantánea $r^2$')
axes[0].set_ylabel('$F(r^2) = P_{\\mathrm{out}}$')
axes[0].set_title('CDF de la potencia Rayleigh')
axes[0].set_xlim([0, 5])
axes[0].legend()

# ── Figura: Outage vs umbral (dB) ───────────────────────────────────────────
gamma_th_dB  = np.linspace(-20, 5, 200)
gamma_th_lin = 10 ** (gamma_th_dB / 10)
p_out_th  = 1 - np.exp(-gamma_th_lin)
p_out_emp = np.array([np.mean(r2 < g) for g in gamma_th_lin])

axes[1].semilogy(gamma_th_dB, p_out_th,  'r-',  lw=2, label='Teórica')
axes[1].semilogy(gamma_th_dB, p_out_emp, 'b--', lw=1.5, label='Empírica')
axes[1].set_xlabel('Umbral $\\gamma_{\\mathrm{th}}$ (dB)')
axes[1].set_ylabel('Probabilidad de interrupción')
axes[1].set_title('Outage probability vs umbral')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Calcular umbral para P_out = 1% ─────────────────────────────────────────
p_target = 0.01
gamma_1pct = -np.log(1 - p_target)           # teórico
print(f'Umbral para P_out = 1%: {10*np.log10(gamma_1pct):.1f} dB  (teórico)')
print(f'                        {np.percentile(r2, 1):.4f} (sim) '
      f'= {10*np.log10(np.percentile(r2, 1)):.1f} dB')

**Preguntas de reflexión:**

1. Para un sistema con sensibilidad de receptor $-90$ dBm y potencia media recibida $-70$ dBm, ¿cuál es la probabilidad de interrupción Rayleigh? ¿Qué margen de desvanecimiento hay que añadir para bajarla al 1%?
2. ¿Cómo cambia $P_{\text{out}}$ si el canal es Rician con $K = 5$ dB en lugar de Rayleigh? Argumenta cualitativamente sin calcular.
3. ¿Por qué la curva de outage tiene pendiente negativa en escala semilogarítmica a bajos umbrales pero se aplana hacia el 100% a umbrales altos?

---
## Ejercicio 5 — Caracterización de Canal Multicamino: Urbano vs Rural

> **Tiempo estimado: ~30 min**

En un canal multicamino, cada camino llega con un retardo y una potencia distintos. El conjunto de retardos y potencias se describe mediante el **perfil de potencia en retardo** (*Power Delay Profile*, PDP). De ese perfil se derivan dos parámetros clave que determinan el comportamiento del canal:

| Parámetro | Origen | Consecuencia |
|-----------|--------|--------------|
| Dispersión de retardo $\sigma_\tau$ | PDP | Ancho de banda coherente $B_c \approx 1/(5\sigma_\tau)$ |
| Desplazamiento Doppler $f_{D,\text{max}}$ | Velocidad | Tiempo coherente $T_c \approx 0.423/f_{D,\text{max}}$ |

Compararemos dos escenarios:

- **Urbano (UMa NLOS)**: muchos caminos con retardos largos → $\sigma_\tau$ grande → $B_c$ pequeño. Sin LOS → distribución Rayleigh.
- **Rural (RMa LOS)**: pocos caminos con retardos cortos → $\sigma_\tau$ pequeño → $B_c$ grande. Con LOS → distribución Rician.

El ejercicio tiene tres partes:

**Parte A** — Simular los PDPs, calcular $\sigma_\tau$ empíricamente y determinar si un sistema de 10 MHz ve *fading plano* o *selectivo en frecuencia* en cada escenario.

**Parte B** — Calcular $f_{D,\text{max}}$ y $T_c$ para distintas velocidades y comparar con la duración del símbolo OFDM.

**Parte C** — Calcular y comparar las curvas BER vs SNR para Rayleigh (urbano) y Rician $K=10$ dB (rural), y cuantificar cuántos dB gana el escenario LOS.

In [ ]:
from scipy.special import erfc

Q = lambda x: 0.5 * erfc(x / np.sqrt(2))

# ══════════════════════════════════════════════════════════════════════════════
# PARTE A — Power Delay Profile y ancho de banda coherente
# ══════════════════════════════════════════════════════════════════════════════

fc_GHz = 2.0    # frecuencia portadora [GHz]
BW_MHz = 10.0   # ancho de banda del sistema [MHz]

# Urbano UMa NLOS — perfil exponencial con múltiples caminos tardíos
tau_urban = np.array([0,  100, 200,  400,  800, 1600])   # retardos [ns]
P_urban   = np.array([0,  -2,  -4,   -8,  -14,  -22])   # potencias [dB]
P_urban   = 10 ** (P_urban / 10)
P_urban  /= P_urban.sum()

# Rural RMa LOS — perfil compacto con pocos caminos
tau_rural = np.array([0,  30,  80,  160])
P_rural   = np.array([0,  -3,  -8,  -18])
P_rural   = 10 ** (P_rural / 10)
P_rural  /= P_rural.sum()

def rms_delay_spread(tau_ns, P):
    """RMS delay spread de un PDP discreto (P normalizado)."""
    tau_mean = np.sum(P * tau_ns)
    return np.sqrt(np.sum(P * tau_ns**2) - tau_mean**2)

sigma_urban = rms_delay_spread(tau_urban, P_urban)
sigma_rural = rms_delay_spread(tau_rural, P_rural)
Bc_urban    = 1 / (5 * sigma_urban * 1e-9) / 1e6   # [MHz]
Bc_rural    = 1 / (5 * sigma_rural * 1e-9) / 1e6   # [MHz]

print('═' * 58)
print('  PARTE A — Dispersión de retardo y ancho de banda coherente')
print('═' * 58)
for name, tau, P, sigma, Bc in [
    ('Urbano (UMa NLOS)', tau_urban, P_urban, sigma_urban, Bc_urban),
    ('Rural  (RMa LOS) ', tau_rural, P_rural, sigma_rural, Bc_rural),
]:
    regime = 'Selectivo en frecuencia  ⚠' if BW_MHz > Bc else 'Fading plano              ✓'
    print(f'\n  {name}')
    print(f'    σ_τ = {sigma:.0f} ns   →   B_c ≈ {Bc:.2f} MHz')
    print(f'    BW = {BW_MHz} MHz   →   {regime}')

# Figura: PDPs comparados
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, tau, P, name, color in [
    (axes[0], tau_urban, P_urban, 'Urbano UMa NLOS', 'steelblue'),
    (axes[1], tau_rural, P_rural, 'Rural  RMa LOS',  'darkorange'),
]:
    w = tau[1] * 0.4 if len(tau) > 1 else 15
    ax.bar(tau, P, width=w, color=color, alpha=0.85, edgecolor='k', linewidth=0.5)
    ax.set_xlabel('Retardo τ (ns)')
    ax.set_ylabel('Potencia relativa')
    ax.set_title(f'PDP — {name}')
plt.suptitle('Power Delay Profiles: Urbano vs Rural', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# PARTE B — Dispersión Doppler y tiempo coherente
# ══════════════════════════════════════════════════════════════════════════════

c       = 3e8
lam     = c / (fc_GHz * 1e9)          # longitud de onda [m]
T_sym_s = 1 / 15e3                    # símbolo OFDM LTE (sin CP): 1/Δf = 66.7 µs

print('\n' + '═' * 58)
print('  PARTE B — Doppler y tiempo coherente')
print('═' * 58)
print(f'\n  f_c = {fc_GHz} GHz   λ = {lam*100:.1f} cm   T_sym = {T_sym_s*1e6:.1f} µs\n')

for name, v_kmh in [('Urbano', 50), ('Rural/autopista', 120)]:
    v_ms   = v_kmh / 3.6
    fD_max = v_ms / lam
    Tc_s   = 0.423 / fD_max
    slots  = Tc_s / T_sym_s
    regime = 'Slow fading ✓' if T_sym_s < Tc_s else 'Fast fading ⚠'
    print(f'  {name} ({v_kmh} km/h):')
    print(f'    f_D,max = {fD_max:.1f} Hz   T_c = {Tc_s*1e3:.1f} ms')
    print(f'    T_sym ({T_sym_s*1e6:.1f} µs) << T_c ({Tc_s*1e3:.1f} ms)   →   {regime}')
    print(f'    Canal válido durante ≈ {slots:.0f} símbolos OFDM consecutivos\n')

# ══════════════════════════════════════════════════════════════════════════════
# PARTE C — Curvas BER: Rayleigh (urbano) vs Rician K=10 dB (rural)
# ══════════════════════════════════════════════════════════════════════════════

SNR_dB  = np.arange(0, 31, 2)
SNR_lin = 10 ** (SNR_dB / 10)
N_bits  = 300_000

bits = rng.integers(0, 2, N_bits)
s    = 2 * bits - 1                  # BPSK: {0,1} → {-1,+1}

# Rayleigh teórica
ber_ray_th = 0.5 * (1 - np.sqrt(SNR_lin / (2 + SNR_lin)))

# Parámetros canal Rician K=10 dB con potencia media unitaria
K_dB   = 10.0
K_lin  = 10 ** (K_dB / 10)
sigma2 = 1 / (2 * (K_lin + 1))
sigma  = np.sqrt(sigma2)
A      = np.sqrt(K_lin / (K_lin + 1))   # amplitud componente LOS

sigma_ch = 1 / np.sqrt(2)              # Rayleigh: σ tal que E[|h|²]=1

ber_ray_mc  = np.zeros(len(SNR_dB))
ber_ric_mc  = np.zeros(len(SNR_dB))

for idx, snr in enumerate(SNR_lin):
    sigma_n = np.sqrt(1.0 / snr)

    # Rayleigh MC (urbano NLOS)
    h_ray   = rng.normal(0, sigma_ch, N_bits) + 1j * rng.normal(0, sigma_ch, N_bits)
    y_ray   = np.abs(h_ray) * s + sigma_n * rng.normal(0, 1, N_bits)
    ber_ray_mc[idx] = np.mean((y_ray > 0).astype(int) != bits)

    # Rician MC (rural LOS, φ_LOS=0 → µ_I=A, µ_Q=0)
    h_ric   = rng.normal(A, sigma, N_bits) + 1j * rng.normal(0, sigma, N_bits)
    y_ric   = np.abs(h_ric) * s + sigma_n * rng.normal(0, 1, N_bits)
    ber_ric_mc[idx] = np.mean((y_ric > 0).astype(int) != bits)

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(SNR_dB, ber_ray_th,  'b-',   lw=2,   label='Rayleigh K=0 — teórica (urbano NLOS)')
ax.semilogy(SNR_dB, ber_ray_mc,  'b--^', lw=1.5, ms=5, label='Rayleigh — Monte Carlo')
ax.semilogy(SNR_dB, ber_ric_mc,  'r-s',  lw=2,   ms=6, label=f'Rician K={K_dB:.0f} dB — Monte Carlo (rural LOS)')
ax.set_xlabel('SNR $\\bar{\\gamma}$ (dB)')
ax.set_ylabel('BER')
ax.set_title('BER vs SNR — Rayleigh (urbano NLOS) vs Rician (rural LOS)')
ax.set_ylim([1e-5, 1])
ax.legend()
plt.tight_layout()
plt.show()

# Penalización Rayleigh vs Rician a BER = 1e-2
target_ber = 1e-2
# Interpolación logarítmica para encontrar el SNR requerido
from scipy.interpolate import interp1d
snr_ray = interp1d(np.log10(np.clip(ber_ray_mc, 1e-6, 1)), SNR_dB, fill_value='extrapolate')(np.log10(target_ber))
snr_ric = interp1d(np.log10(np.clip(ber_ric_mc, 1e-6, 1)), SNR_dB, fill_value='extrapolate')(np.log10(target_ber))
print(f'\n  PARTE C — Penalización a BER = {target_ber:.0e}:')
print(f'    Rayleigh requiere {snr_ray:.1f} dB')
print(f'    Rician K={K_dB:.0f} dB requiere {snr_ric:.1f} dB')
print(f'    → Ganancia LOS: {snr_ray - snr_ric:.1f} dB')

**Preguntas de reflexión:**

1. **Parte A** — El escenario urbano tiene $B_c \approx 0{,}67\,\text{MHz}$ y el sistema usa 10 MHz. ¿Qué implica esto para un sistema de banda ancha? ¿Qué técnica de transmisión usaría LTE/5G para lidiar con el fading selectivo en frecuencia?

2. **Parte B** — A 50 km/h el canal es válido durante varios cientos de símbolos OFDM. ¿Qué ventaja ofrece esto para la estimación del canal? ¿Qué ocurriría si aumentaras la velocidad a 300 km/h (tren de alta velocidad)?

3. **Parte C** — La curva Rician $K=10\,\text{dB}$ cae más rápido que Rayleigh a alta SNR. Explica cualitativamente por qué: ¿cómo afecta la componente LOS a la probabilidad de deep fading? ¿Hacia qué curva converge la BER Rician cuando $K \to \infty$?

---
## Ejercicio 6 — Caracterización de Canal con TR 38.901

> **Tiempo estimado: ~20 min**

Dado un escenario de despliegue 5G NR, usaremos los parámetros de **3GPP TR 38.901** para caracterizar completamente el canal: coherence bandwidth, coherence time, régimen de fading y tipo de distribución de envolvente.

**Escenario**: UMi Street Canyon **LOS**, $f_c = 3{,}5\,\text{GHz}$, $v = 30\,\text{km/h}$, $\sigma_\tau = 50\,\text{ns}$, factor Rician $K = 9\,\text{dB}$.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Parámetros TR 38.901 UMi LOS ────────────────────────────────────────────
fc_GHz    = 3.5           # frecuencia portadora [GHz]
v_kmh     = 30.0          # velocidad del terminal [km/h]
sigma_tau = 50e-9         # delay spread RMS [s]
K_dB      = 9.0           # factor K de Rician [dB]
Bs_MHz    = 40.0          # ancho de banda del sistema [MHz]
Ts_ms     = 0.5           # duración de ranura NR μ=1 [ms]

# ── 1. Coherence bandwidth ───────────────────────────────────────────────────
Bc_Hz = 1 / (5 * sigma_tau)
Bc_MHz = Bc_Hz / 1e6
Delta_f_kHz = 30.0        # espaciado de subportadora NR μ=1 [kHz]

print('═' * 55)
print('  Caracterización de Canal — TR 38.901 UMi LOS')
print('═' * 55)
print(f'\n1. DISPERSIÓN TEMPORAL')
print(f'   σ_τ = {sigma_tau*1e9:.0f} ns')
print(f'   B_c ≈ 1/(5σ_τ) = {Bc_MHz:.2f} MHz')
print(f'   Δf subportadora (μ=1) = {Delta_f_kHz:.0f} kHz')
regime_sub = 'FLAT fading' if Delta_f_kHz*1e3 < Bc_Hz else 'Frequency-selective'
regime_sys = 'Frequency-selective fading' if Bs_MHz*1e6 > Bc_Hz else 'Flat fading'
print(f'   → Por subportadora ({Delta_f_kHz} kHz << {Bc_MHz:.0f} MHz): {regime_sub}')
print(f'   → Sistema completo  ({Bs_MHz:.0f} MHz >> {Bc_MHz:.0f} MHz): {regime_sys}')
print(f'   → Cyclic prefix mínimo recomendado: ≥ σ_τ = {sigma_tau*1e9:.0f} ns')

# ── 2. Coherence time ────────────────────────────────────────────────────────
c = 3e8
v_ms   = v_kmh / 3.6
lam    = c / (fc_GHz * 1e9)
fD_max = v_ms / lam
Tc_ms  = 0.423 / fD_max * 1e3   # en ms

print(f'\n2. VARIACIÓN TEMPORAL (DOPPLER)')
print(f'   v = {v_kmh} km/h = {v_ms:.2f} m/s,  λ = {lam*100:.1f} cm')
print(f'   f_D,max = v/λ = {fD_max:.1f} Hz')
print(f'   T_c ≈ 0.423/f_D,max = {Tc_ms:.2f} ms')
print(f'   Duración de ranura NR (μ=1): {Ts_ms} ms')
regime_time = 'SLOW fading' if Ts_ms < Tc_ms else 'Fast fading'
print(f'   → T_slot = {Ts_ms} ms << T_c = {Tc_ms:.2f} ms: {regime_time}')
slots_per_Tc = Tc_ms / Ts_ms
print(f'   → Canal válido durante ≈ {slots_per_Tc:.0f} ranuras consecutivas')

# ── 3. Distribución de la envolvente ────────────────────────────────────────
K_lin = 10 ** (K_dB / 10)
frac_LOS    = K_lin / (K_lin + 1) * 100
frac_scatter = 100 - frac_LOS

print(f'\n3. DISTRIBUCIÓN DE LA ENVOLVENTE')
print(f'   K = {K_dB} dB → K_lineal = {K_lin:.1f}')
print(f'   Potencia LOS:      {frac_LOS:.1f}%')
print(f'   Potencia dispersa: {frac_scatter:.1f}%')
print(f'   → Modelo: RICIAN (LOS dominante, deep fading poco probable)')

# ── 4. Figura comparativa: PDF Rician K=9dB vs Rayleigh ─────────────────────
sigma2_R  = 0.5                          # Rayleigh: potencia unitaria
sigma2_Ri = 1 / (2 * (K_lin + 1))       # Rician: potencia total unitaria
A_Ri      = np.sqrt(K_lin / (K_lin + 1))

r_ax = np.linspace(0, 3, 500)
pdf_rayleigh = rayleigh.pdf(r_ax, scale=np.sqrt(sigma2_R))
pdf_rician   = rice.pdf(r_ax, b=A_Ri / np.sqrt(sigma2_Ri), scale=np.sqrt(sigma2_Ri))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r_ax, pdf_rayleigh, 'b-',  lw=2, label='Rayleigh K=0 (NLOS)')
ax.plot(r_ax, pdf_rician,   'r-',  lw=2, label=f'Rician K={K_dB} dB (LOS)')
ax.axvline(A_Ri, color='r', ls='--', alpha=0.5, label=f'Componente LOS A={A_Ri:.2f}')
ax.set_xlabel('Envolvente $r$')
ax.set_ylabel('Densidad de probabilidad')
ax.set_title(f'Rayleigh vs Rician K={K_dB} dB — UMi LOS 3.5 GHz')
ax.legend()
plt.tight_layout()
plt.show()
print('\nFigura: el pico Rician se desplaza hacia A (componente LOS); cola izquierda')
print('(deep fading) es sustancialmente más pequeña que en Rayleigh.')

**Preguntas de reflexión:**

1. Cambia el escenario a **UMa NLOS** con $\sigma_\tau = 300\,\text{ns}$ y $K = 0$ (Rayleigh). ¿Cómo cambian $B_c$, $T_c$ y el tipo de distribución? ¿Cuántas ranuras NR dura el canal ahora?
2. Con $\sigma_\tau = 50\,\text{ns}$ y subportadoras de 30 kHz, ¿el cyclic prefix de 2,34 µs de NR (μ=1) es suficiente para eliminar la ISI? Justifica con los números.
3. Si el terminal acelera a $v = 120\,\text{km/h}$, ¿cuántos slots NR caben dentro de $T_c$? ¿El canal sigue siendo slow fading?

---
## Conclusiones

En este laboratorio hemos:

- **Verificado** que la suma de componentes I/Q gaussianas produce una envolvente Rayleigh — conectando la derivación matemática del index.md con su implementación numérica.
- **Observado** cómo el factor K de Rician desplaza el diagrama fasorial lejos del origen, reduciendo la probabilidad de deep fading.
- **Cuantificado** la penalización por Rayleigh fading en la curva BER vs SNR y verificado la fórmula cerrada con Monte Carlo.
- **Calculado** la outage probability y el fade margin necesario para distintos objetivos de disponibilidad.
- **Caracterizado** un canal multicamino en escenarios urbano y rural: calculamos $\sigma_\tau$, $B_c$, $T_c$ desde el PDP y comparamos la BER de Rayleigh (NLOS) y Rician (LOS).
- **Aplicado** los parámetros de TR 38.901 para caracterizar completamente un canal 5G NR real.

Estos resultados motivan las técnicas que estudiaremos a lo largo del curso: OFDM (Sesión 03), codificación de canal (Sesión 04) y sistemas MIMO (Sesión 06).

---

### Recursos Adicionales

- T. S. Rappaport, *Wireless Communications*, 2nd ed. — Cap. 3 y 5
- A. Goldsmith, *Wireless Communications*, Cambridge, 2005 — Cap. 3
- 3GPP TR 38.901 — *Study on channel model for frequencies from 0.5 to 100 GHz*
- scipy.stats.rayleigh: [documentación](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rayleigh.html)